# KOD ANALIZUJE TABELĘ CSV I ZAPISUJE W CZYSTEJ FORMIE ORAZ TWORZY PLIK (*NAZWA*_clean_for_rag.csv) 


### 1. ŚCIEŻKA DO WEJŚCIOWEGO CSV (ZMIENIASZ TYLKO TO)
### csv_path = r"<ŚCIEŻKA DO PLIKU>\<NAZWAPLIKU>.csv" 

In [ ]:
# === KOMPLEKSOWE, ELASTYCZNE BADANIE FAQ / CZAT CSV + ZAPIS POD RAG ===

import pandas as pd                                                                 #    <=========I
import re                                                                           #    <=========I
import os                                                                           #    <=========I
                                                                                    #    <=========I
# 1. ŚCIEŻKA DO WEJŚCIOWEGO CSV (ZMIENIASZ TYLKO TO)                                #    <=========I
csv_path = r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv"     #    <=========I  MUSISZ WSKAZAĆ SWÓJ PLIK CSV Z FAQ / CZATEM

# 2. Wczytanie danych (obsługa polskich znaków, autodetekcja separatora)
df = pd.read_csv(
    csv_path,
    encoding="utf-8",   # jeśli będą krzaki, zmień na "cp1250"
    engine="python",
    sep=None,           # autodetekcja separatora
)

# 3. Ustawienia wyświetlania (czytelność)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

print("KOLUMNY W DF:", list(df.columns))

# 4. Automatyczna konfiguracja kolumn FAQ / czatu (odporna na nazwy)
cols_lower = {c.lower().strip(): c for c in df.columns}

def find_col(candidates):
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

QA_COLS = {
    "shop": find_col(["shop", "sklep"]),
    "url":  find_col(["source_url", "url", "link", "href"]),
    "q":    find_col(["question", "pytanie", "query", "user_message"]),
    "a":    find_col(["answer", "odpowiedz", "odpowiedź", "response", "bot_message"]),
}

print("ZMAPOWANE KOLUMNY QA_COLS:", QA_COLS)

# 5. Funkcja czyszcząca tekst (HTML, encje, whitespace)
def clean_text(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s)

    # usuń proste znaczniki HTML
    s = re.sub(r"<[^>]+>", " ", s)

    # podstawowe encje HTML
    html_entities = {
        "&nbsp;": " ",
        "&amp;": "&",
        "&quot;": '"',
        "&apos;": "'",
        "&lt;": "<",
        "&gt;": ">",
    }
    for ent, rep in html_entities.items():
        s = s.replace(ent, rep)

    # twarde spacje, CR
    s = s.replace("\xa0", " ")
    s = s.replace("\r", " ")

    # wielokrotne białe znaki → jedna spacja
    s = re.sub(r"\s+", " ", s)

    # przycięcie
    return s.strip()

# 6. Czyszczenie tekstu we wszystkich kolumnach tekstowych (pełna elastyczność)
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].apply(clean_text)
    df.loc[df[col].astype(str).str.strip() == "", col] = pd.NA

# 7. Podstawowe informacje (na całym df)
print("=" * 60)
print("1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)")
print("=" * 60)
print(f"Liczba wierszy: {df.shape[0]}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"Całkowita liczba obserwacji: {df.shape[0] * df.shape[1]}")

# 8. Nazwy kolumn
print("=" * 60)
print("2. NAZWY KOLUMN")
print("=" * 60)
print(df.columns.tolist())

# 9. Podgląd: 3 pierwsze i 3 ostatnie wiersze
print("=" * 60)
print("3. PODGLĄD DANYCH (pierwsze 3 wiersze)")
print("=" * 60)
display(df.head(3).reset_index(drop=True))

print("=" * 60)
print("4. PODGLĄD DANYCH (ostatnie 3 wiersze)")
print("=" * 60)
display(df.tail(3).reset_index(drop=True))

# 10. Typy danych i braki – na całym df
print("=" * 60)
print("5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)")
print("=" * 60)
info_df = pd.DataFrame({
    "Typ danych": df.dtypes,
    "Liczba brakujących": df.isnull().sum(),
    "% brakujących": (df.isnull().sum() / len(df) * 100).round(2),
    "Unikalne wartości": df.nunique()
})
display(info_df.reset_index().rename(columns={"index": "Kolumna"}))

# 11. Statystyki numeryczne – całe df (jeśli są)
print("=" * 60)
print("6. STATYSTYKI OPISOWE (kolumny numeryczne, WSZYSTKIE REKORDY)")
print("=" * 60)
num = df.select_dtypes(include=["int64", "float64"])
if not num.empty:
    display(num.describe())
else:
    print("Brak kolumn numerycznych.")

# 12. Statystyki tekstowe – całe df
print("=" * 60)
print("7. STATYSTYKI TEKSTOWE (WSZYSTKIE REKORDY)")
print("=" * 60)
display(df.describe(include=["object", "string"]))

# 13. Zakres danych – całe df
print("=" * 60)
print("8. ZAKRES DANYCH (WSZYSTKIE REKORDY)")
print("=" * 60)
for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: {df[col].min()} -> {df[col].max()}")
    elif df[col].dtype == "object":
        print(f"{col}: {df[col].nunique()} unikalnych wartości")

# 14. Specjalne EDA dla FAQ / czatu – tylko jeśli mamy Q/A
has_q = QA_COLS["q"] is not None
has_a = QA_COLS["a"] is not None
has_shop = QA_COLS["shop"] is not None
has_url = QA_COLS["url"] is not None

if has_q and has_a:
    print("=" * 60)
    print("A1. INFO O PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    n_rows = len(df)
    n_shops = df[QA_COLS["shop"]].nunique() if has_shop else None
    n_urls  = df[QA_COLS["url"]].nunique() if has_url else None
    n_q     = df[QA_COLS["q"]].nunique()
    n_a     = df[QA_COLS["a"]].nunique()

    print(f"Liczba rekordów:              {n_rows}")
    if has_shop:
        print(f"Liczba unikalnych sklepów:    {n_shops}")
    if has_url:
        print(f"Liczba unikalnych URL-i:      {n_urls}")
    print(f"Liczba unikalnych PYTAŃ:      {n_q}")
    print(f"Liczba unikalnych ODPOWIEDZI: {n_a}")

    # duplikaty – globalnie
    print("=" * 60)
    print("A2. DUPLIKATY PYTAŃ I PAR Q&A (WSZYSTKIE REKORDY)")
    print("=" * 60)

    dup_q  = df.duplicated(subset=[QA_COLS["q"]]).sum()
    dup_qa = df.duplicated(subset=[QA_COLS["q"], QA_COLS["a"]]).sum()

    print(f"Liczba zduplikowanych PYTAŃ:   {dup_q}")
    print(f"Liczba zduplikowanych PAR Q&A: {dup_qa}")

    # długości – globalnie
    print("=" * 60)
    print("A3. DŁUGOŚCI PYTAŃ I ODPOWIEDZI (WSZYSTKIE REKORDY)")
    print("=" * 60)

    df["question_len"] = df[QA_COLS["q"]].astype(str).str.len()
    df["answer_len"]   = df[QA_COLS["a"]].astype(str).str.len()

    print("Statystyki długości PYTAŃ:")
    print(df["question_len"].describe())

    print("\nStatystyki długości ODPOWIEDZI:")
    print(df["answer_len"].describe())

    print("\nNajkrótsze PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajkrótsze ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    # braki – globalnie
    print("=" * 60)
    print("A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    missing_q = df[QA_COLS["q"]].isna().sum()
    missing_a = df[QA_COLS["a"]].isna().sum()

    print(f"Brakujące (NaN) PYTANIA:    {missing_q}")
    print(f"Brakujące (NaN) ODPOWIEDZI: {missing_a}")

    bad_rows = df[df[QA_COLS["q"]].isna() | df[QA_COLS["a"]].isna()]
    print("\nPrzykładowe rekordy z brakami (max 3):")
    display(bad_rows.head(3).reset_index(drop=True))

else:
    print("=" * 60)
    print("A*. BRAK PEŁNEGO ZESTAWU KOLUMN Q/A – pomijam sekcję FAQ")
    print("    (potrzebne kolumny logiczne: question + answer)")
    print("=" * 60)

print("=" * 60)
print("=== BADANIE FAQ ZAKOŃCZONE ===")
print("=" * 60)

# 15. DYNAMICZNY ZAPIS OCZYSZCZONEGO PLIKU POD RAG / EMBEDDINGI

base_name = os.path.basename(csv_path)           # np. 'faq_output.csv'
name_no_ext, ext = os.path.splitext(base_name)   # 'faq_output', '.csv'
out_dir = os.path.dirname(csv_path)

out_filename_rag  = f"{name_no_ext}__clean_for_rag{ext}"
out_filename_full = f"{name_no_ext}__clean_full{ext}"

out_path_rag  = os.path.join(out_dir, out_filename_rag)
out_path_full = os.path.join(out_dir, out_filename_full)

cols_for_rag = []

if QA_COLS.get("q") is not None:
    cols_for_rag.append(QA_COLS["q"])
if QA_COLS.get("a") is not None:
    cols_for_rag.append(QA_COLS["a"])
if QA_COLS.get("url") is not None:
    cols_for_rag.append(QA_COLS["url"])
if QA_COLS.get("shop") is not None:
    cols_for_rag.append(QA_COLS["shop"])

if cols_for_rag:
    df_rag = df[cols_for_rag].copy()
    df_rag = df_rag.dropna(subset=[QA_COLS["q"], QA_COLS["a"]], how="any")
    df_rag = df_rag.reset_index(drop=True)
    df_rag.to_csv(out_path_rag, index=False, encoding="utf-8")
    print(f"Oczyszczony plik pod RAG zapisany do:\n{out_path_rag}")
    print(f"Liczba zapisanych par Q&A: {len(df_rag)}")
else:
    df.to_csv(out_path_full, index=False, encoding="utf-8")
    print("Nie znaleziono pełnego zestawu kolumn Q/A.")
    print(f"Zapisano pełny oczyszczony DataFrame do:\n{out_path_full}")
    

KOLUMNY W DF: ['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
ZMAPOWANE KOLUMNY QA_COLS: {'shop': None, 'url': None, 'q': 'Pytanie', 'a': 'Odpowiedź'}
1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)
Liczba wierszy: 44
Liczba kolumn: 5
Całkowita liczba obserwacji: 220
2. NAZWY KOLUMN
['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
3. PODGLĄD DANYCH (pierwsze 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,FORTE Meble,https://forte.com.pl,Dystrybucja i zakup,Gdzie mogę zakupić meble marki FORTE?,Meble FORTE sprzedawane są przez sieć partnerską w ponad 1000 salonów meblowych na terenie całej Polski. Pełną listę punktów sprzedaży można znaleźć na naszej stronie internetowej w zakładce 'Gdzi...
1,FORTE Meble,https://forte.com.pl,Bezpieczeństwo i montaż,Czy meble FORTE muszą być mocowane do ściany?,"Tak, dla bezpieczeństwa użytkowników (szczególnie dzieci) większość wysokich mebli, takich jak komody, regały czy szafy, posiada w zestawie specjalne okucia przeznaczone do montażu ściennego. Kate..."
2,Szynaka Meble,https://szynaka.pl/pytania-i-odpowiedzi/,Zakup i Dostępność,Gdzie można kupić meble?,Produkty firmy Szynaka Meble można kupić w ponad 350 salonach meblowych w całej Polsce. Sklep znajdujący się najbliżej Twojego miejsca zamieszkania znajdziesz w zakładce „Gdzie kupić?”.


4. PODGLĄD DANYCH (ostatnie 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,Dekoria,https://dekoria.pl,Darmowe próbki,Czy można zamówić próbki tkanin przed zakupem i ile one kosztują?,"Tak, na stronie dekoria.pl/probnik można wybrać i zamówić do 10 darmowych próbek interesujących Cię tkanin. Wysyłamy je bezpłatnie za pośrednictwem Poczty Polskiej (list zwykły), aby ułatwić dopas..."
1,Dekoria,https://dekoria.pl,Faktury i prawo do zwrotu,Czy zamawiając towar na firmę (faktura VAT) przysługuje mi prawo do zwrotu?,"Nie. Towar zakupiony w ramach prowadzonej działalności gospodarczej (z podaniem numeru NIP firmy) nie podlega standardowemu konsumenckiemu prawu do zwrotu bez podania przyczyny, ponieważ kupujący ..."
2,Fabryka Form,https://fabrykaform.pl,Oryginalność produktów,Czy wszystkie produkty oferowane w Fabryce Form są produktami oryginalnymi?,"Tak, jesteśmy oficjalnym dystrybutorem wszystkich marek premium prezentowanych w naszym sklepie (m.in. Alessi, Stelton, Joseph Joseph). Wszystkie towary są w 100% oryginalne, fabrycznie nowe i obj..."


5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)


...


Najdłuższe ODPOWIEDZI (top 3):


,Odpowiedź,answer_len
0,"Reklamacji podlegają m.in. uszkodzenia transportowe, rysy, pęknięcia, nieszczelności urządzeń AGD lub niezgodność z opisem. Wymagany jest dowód zakupu (paragon/faktura lub potwierdzenie płatności ...",306
1,"Standardowo kurier ma obowiązek wniesienia przesyłki pod drzwi lokalu, jeżeli jej waga nie przekracza 30 kg. W przypadku paczek cięższych (np. duże szafy, łóżka, narożniki) kurier dostarcza przesy...",301
2,"Marka Kler oferuje bardzo szerokie możliwości personalizacji modułowej. Wiele kolekcji pozwala na dobór układu elementów, rodzaju i koloru skóry lub tkaniny premium, a także funkcji dodatkowych (n...",299


A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)
Brakujące (NaN) PYTANIA:    0
Brakujące (NaN) ODPOWIEDZI: 0

Przykładowe rekordy z brakami (max 3):


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź,question_len,answer_len


=== BADANIE FAQ ZAKOŃCZONE ===
Oczyszczony plik pod RAG zapisany do:
C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output__clean_for_rag.csv
Liczba zapisanych par Q&A: 44
